In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib as mpl
from itertools import product

In [45]:
data = pd.read_pickle('data/raw/data-maize-soya.pkl')
data = data.replace({'none': None, 'None': None, 'NONE': None, 'NAN': None, 'NaN': None})
# Drop rows with no production data
# data = data.loc[data.maizeprodkg_ha.notna()]
metadata = pd.read_csv('data/raw/05_FMBA24sf2_meta_250620.csv', sep='\t')
metadata = metadata.set_index('colname')

In [46]:
# Check if there are records with both maize and soya production data
data[data.maizeprodkg_ha.notna() & data.soyaprodkg_ha.notna()]

,maizeprodkg_ha,maizemarketing,maizeharvest_seperate,maizeharvestsheller,maizevariety,maizerecycledseed,maizefieldprep,maizeplantingmethod,maizeplantmonth,maizeplantmonthpart,...,estimated_sos,estimated_eos,estimated_los,peak_ndvi_time,peak_ndvi_value,sos_ndvi_value,eos_ndvi_value,ndvi_array_j,elevation,innacessibility_index
fieldID,,,,,,,,,,,,,,,,,,,,,
W0Np8wggre,804.5738,selleat,yes,yes,DKC9089,recycled,handhoe,different,December,first,...,2023-12-16,2024-08-01,229.0,2024-03-23,0.767944,0.333138,0.29229,3149.0,1087.0,0.796547


In [47]:
# We will drop that record and we'll focus on records that have either maize or soya
# that way we can easily create a single production column
data = data[~(data.maizeprodkg_ha.notna() & data.soyaprodkg_ha.notna())]
(data.maizeprodkg_ha.notna() & data.soyaprodkg_ha.notna()).any()

np.False_

# Fertilizer data
Some of the Fertilizer information is missing. Let's see if we can fill that in.

In [48]:
# Assign amount of zero, when no fertilizer was applied
farmer_applied_fertilizer = data[f'basal_application'].notna()
for element in ('N', 'P', 'K', ''):
    data.loc[~farmer_applied_fertilizer, f'KG{element}basal_ha'] = 0

for i in range(1, 4):
    farmer_applied_fertilizer = data[f'top{i}application'].notna()
    for element in ('N', 'P', 'K', ''):
        data.loc[~farmer_applied_fertilizer, f'KG{element}top{i}_ha'] = 0

In [49]:
# Re-calculate total amounts
for element in ('N', 'P', 'K', ''):
    data[f'KG{element}total_ha'] = \
        data[[f'KG{element}basal_ha']+[f'KG{element}top{i}_ha' for i in range(1,4)]].sum(axis=1, skipna=False)

In [50]:
print('Records where fertilizer was applied but amount was not recorded')
farmer_applied_fertilizer = data[f'basal_application'].notna()
applied_amount_recorded = data[f'KGbasal_ha'].notna()
print(f'Basal: {(farmer_applied_fertilizer & ~applied_amount_recorded).sum()}/{len(data)}')
for i in range(1, 4):
    farmer_applied_fertilizer = data[f'top{i}application'].notna()
    applied_amount_recorded = data[f'KGtop{i}_ha'].notna()
    print(f'Top {i}: {(farmer_applied_fertilizer & ~applied_amount_recorded).sum()}/{len(data)}')

Records where fertilizer was applied but amount was not recorded
Basal: 66/2573
Top 1: 882/2573
Top 2: 596/2573
Top 3: 10/2573


In [51]:
data[[f'KG{e}total_ha' for e in ('N', 'P', 'K', '')]].describe()

,KGNtotal_ha,KGPtotal_ha,KGKtotal_ha,KGtotal_ha
count,1688.000000,1688.000000,1688.000000,1688.000000
mean,68.187675,16.865672,0.115997,234.351240
std,70.172508,21.479305,2.034576,241.853764
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,66.838617,0.000000,0.000000,232.343735
75%,116.559292,27.540063,0.000000,390.458070
max,401.262895,154.702326,58.954456,1253.946546


In [52]:
print('Records where fertilizer was applied but timing was not recorded')
for i in range(1, 4):
    farmer_applied_fertilizer = data[f'top{i}application'].notna()
    timing_recorded = data[f'top{i}timing'].notna()
    print(f'Top {i}: {(farmer_applied_fertilizer & ~timing_recorded).sum()}/{len(data)}')

Records where fertilizer was applied but timing was not recorded
Top 1: 0/2573
Top 2: 0/2573
Top 3: 0/2573


In [53]:
data.to_pickle('data/maize-soya-clean.pkl')